# Train the proxy reward model (Phase 1)

Runs `src/train_rm.py` on a Colab T4 (plain AdamW fits in 15GB). The real-run flags below mirror `configs/rm.yaml`.

**Flow:** GPU check -> mount Drive -> locate repo + pool -> deps -> **smoke test** (8/8 questions, 1 epoch) -> **real run** -> read metrics.

**Bar for the real run:** held-out **AUROC ~0.75-0.85**, not raw acc. Majority baseline is ~0.67 at pos_rate 0.327, so acc alone lies. AUROC creeping toward 0.95 = RM too strong, nothing to over-optimize against.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# EDIT THIS to wherever the repo lives in Drive (must contain src/, configs/, data/rm_pool.jsonl).
REPO_DIR = '/content/drive/MyDrive/reward-overoptimization'

assert os.path.isdir(REPO_DIR), f'REPO_DIR not found: {REPO_DIR}'
os.chdir(REPO_DIR)  # python -m src.train_rm and the relative data/ results/ paths need repo root as cwd
print('cwd =', os.getcwd())
for p in ['src/train_rm.py', 'src/rm.py', 'src/data.py', 'configs/rm.yaml', 'data/rm_pool.jsonl']:
    print(('OK  ' if os.path.exists(p) else 'MISS'), p)

In [ ]:
!pip install -q -U "transformers>=4.44" scikit-learn
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

## Smoke test

8 train / 8 eval questions, 1 epoch, into `results/rm_smoke/`.

In [ ]:
!python -m src.train_rm --train_questions 8 --eval_questions 8 --epochs 1 --out_dir results/rm_smoke

In [ ]:
import json
print(json.dumps(json.load(open('results/rm_smoke/metrics.json'))['history'], indent=2))

## Real run

Flags mirror `configs/rm.yaml` (400 train / 400 eval questions, 2 epochs). `pos_weight` is left off so BCE stays unweighted.

In [ ]:
!python -m src.train_rm \
  --pool data/rm_pool.jsonl \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --train_questions 400 \
  --eval_questions 400 \
  --epochs 2 \
  --batch_size 8 \
  --eval_batch_size 16 \
  --lr 1e-5 \
  --weight_decay 0.0 \
  --warmup_ratio 0.1 \
  --max_grad_norm 1.0 \
  --max_length 768 \
  --dtype bfloat16 \
  --out_dir results/rm_v1 \
  --seed 0

In [ ]:
import json
m = json.load(open('results/rm_v1/metrics.json'))
for h in m['history']:
    print(f"epoch {h['epoch']}: loss={h['train_loss']:.4f}  auroc={h['auroc']:.3f}  "
          f"acc={h['acc']:.3f} (baseline {h['majority_baseline']:.3f})  "
          f"bal_acc={h['balanced_acc']:.3f}  tpr={h['tpr']:.3f}  tnr={h['tnr']:.3f}")
last = m['history'][-1]['auroc']
verdict = 'in target band' if 0.75 <= last <= 0.85 else ('TOO STRONG - cut train_questions/epochs' if last > 0.85 else 'too weak - add data/epochs')
print(f'\nfinal AUROC {last:.3f} -> {verdict}')